# 01 – Dataset Inspection, Cleaning, Data Dictionary & Splitting
**Owner:** Member 1 (Data Lead and Coordinator)

This notebook performs the initial audit and preparation of the raw MOF CO₂ adsorption dataset:
1. Inspect raw `train.csv` and `test.csv` (rows, columns, types, missing values, duplicates, units).
2. Standardize column names/types and remove exact duplicates.
3. Handle missing values safely without leakage.
4. Detect statistical outliers using IQR and verify physical validity.
5. Generate a complete `docs/data_dictionary.md`.
6. Add `row_id` and perform a reproducible 70% / 15% / 15% train/val/test split (`RANDOM_SEED = 42`).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Ensure project root is on Python path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CLEAN_DATA_PATH,
    DOCS_DIR,
    ID_COL,
    PROCESSED_DATA_DIR,
    RANDOM_SEED,
    RAW_NAME_COL,
    RAW_TEST_PATH,
    RAW_TRAIN_PATH,
    TARGET_COL,
    TARGET_UNIT_NOTE,
    TEST_PATH,
    TEST_RATIO,
    TRAIN_PATH,
    TRAIN_RATIO,
    VAL_PATH,
    VAL_RATIO,
)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Random Seed : {RANDOM_SEED}")
print(f"Target Col  : {TARGET_COL}")

Project Root: C:\Users\nourh\.gemini\antigravity\scratch\mof-co2-uptake-prediction
Random Seed : 42
Target Col  : CO2 uptake at 0.15 bar and 298K


## 1. Initial Inspection of Raw Dataset Files

In [2]:
raw_train = pd.read_csv(RAW_TRAIN_PATH)
raw_test = pd.read_csv(RAW_TEST_PATH)

print(f"Raw train.csv: {raw_train.shape[0]:,} rows x {raw_train.shape[1]} columns | "
      f"{RAW_TRAIN_PATH.stat().st_size / (1024**2):.2f} MB")
print(f"Raw test.csv : {raw_test.shape[0]:,} rows x {raw_test.shape[1]} columns | "
      f"{RAW_TEST_PATH.stat().st_size / (1024**2):.2f} MB")

print(f"\nIs TARGET_COL in raw train.csv? {TARGET_COL in raw_train.columns}")
print(f"Is TARGET_COL in raw test.csv ? {TARGET_COL in raw_test.columns}")

missing_in_raw_test = [c for c in raw_train.columns if c not in raw_test.columns]
print(f"Columns in raw train.csv but missing in raw test.csv: {len(missing_in_raw_test)}")

raw_train.head(3)

Raw train.csv: 5,014 rows x 328 columns | 13.45 MB
Raw test.csv : 2,000 rows x 273 columns | 4.95 MB

Is TARGET_COL in raw train.csv? True
Is TARGET_COL in raw test.csv ? False
Columns in raw train.csv but missing in raw test.csv: 55


,MOFname,ASA [m^2/cm^3],Df,Di,Dif,NASA [m^2/cm^3],POAV [cm^3/g],POAVF,PONAV [cm^3/g],PONAVF,...,sum-f-lig-I-3,sum-f-lig-T-0,sum-f-lig-T-1,sum-f-lig-T-2,sum-f-lig-T-3,sum-f-lig-S-0,sum-f-lig-S-1,sum-f-lig-S-2,sum-f-lig-S-3,CO2 uptake at 0.15 bar and 298K
0,XAGCUE_clean,0.00,2.87682,6.41175,6.41175,487.410,0.000000,0.0000,0.101882,0.21558,...,2496.0,2784.0,7296.0,10752.0,10752.0,242.0856,523.7232,844.9008,1049.6448,1.266370
1,SOBZEQ_clean,2298.52,5.44324,7.06044,7.04565,0.000,0.738273,0.6363,0.000000,0.00000,...,496.0,632.0,1560.0,2240.0,2544.0,58.2840,130.2840,194.2144,231.0000,8.224130
2,AVAQIX01_clean,0.00,3.61603,5.36267,5.34980,600.353,0.000000,0.0000,0.226377,0.30062,...,672.0,1024.0,2400.0,3040.0,2688.0,91.7328,204.2656,276.2848,265.7760,3.694178


**Important Observation:** `data/raw/test.csv` does not include `CO2 uptake at 0.15 bar and 298K` (the target variable) and omits 54 constant descriptor columns. Because supervised evaluation requires `y_true`, `data/raw/train.csv` (5,014 labeled MOFs) is used as our working dataset and split into `train.csv` (70%), `val.csv` (15%), and `test.csv` (15%).

## 2. Column Standardization, Data Types, Missing Values & Duplicates

In [3]:
df = raw_train.copy()

# 1. Strip any accidental leading/trailing whitespace in column names
df.columns = [c.strip() for c in df.columns]

# 2. Verify data types (MOFname should be string/object; all others numeric)
non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
print("Non-numeric columns:", non_numeric_cols)
for col in df.columns:
    if col != RAW_NAME_COL:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 3. Missing-value audit before cleaning
missing_counts_raw = df.isna().sum()
missing_cols = missing_counts_raw[missing_counts_raw > 0]
print("\nColumns with missing values in raw_train:")
print(missing_cols)

# 4. Exact duplicate detection & removal
n_exact_dups = int(df.duplicated().sum())
print(f"\nExact duplicate rows found: {n_exact_dups}")
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"Rows after removing exact duplicates: {len(df):,} (Unique {RAW_NAME_COL}: {df[RAW_NAME_COL].nunique():,})")

# 5. Missing-value handling (drop the single row missing Df = 1/5000 = 0.02% to avoid pre-split imputation leakage)
rows_with_na = int(df.isna().any(axis=1).sum())
print(f"Rows containing any NaN: {rows_with_na}")
if rows_with_na > 0:
    print("Dropping incomplete rows:")
    display_cols = [RAW_NAME_COL] + missing_cols.index.tolist()
    print(df.loc[df.isna().any(axis=1), display_cols])
    df = df.dropna().reset_index(drop=True)

# 6. Check for infinite values
num_cols = [c for c in df.columns if c != RAW_NAME_COL]
inf_count = int(np.isinf(df[num_cols].to_numpy()).sum())
print(f"Remaining NaN values: {int(df.isna().sum().sum())} | Inf values: {inf_count}")
print(f"Final cleaned row count: {len(df):,}")

Non-numeric columns:

 ['MOFname']

Columns with missing values in raw_train:
Df    1
dtype: int64

Exact duplicate rows found: 14


Rows after removing exact duplicates: 5,000 (Unique MOFname: 5,000)
Rows containing any NaN: 1
Dropping incomplete rows:
           MOFname  Df
19  FIRNAX01_clean NaN
Remaining NaN values: 0 | Inf values: 0
Final cleaned row count: 4,999


## 3. Physical Validity & IQR Outlier Investigation
We inspect physical bounds on structural properties and target uptake, and compute IQR outlier counts **without automatically deleting valid extreme MOFs**.

In [4]:
structural_cols = [
    "ASA [m^2/cm^3]", "Df", "Di", "Dif", "NASA [m^2/cm^3]",
    "POAV [cm^3/g]", "POAVF", "PONAV [cm^3/g]", "PONAVF",
    "density [g/cm^3]", "total_SA_volumetric", "total_SA_gravimetric",
    "total_POV_volumetric", "total_POV_gravimetric", TARGET_COL,
]

# Physical validity checks
physical_checks = {
    "Negative surface area (ASA)": int((df["ASA [m^2/cm^3]"] < 0).sum()),
    "Negative pore diameter (Df/Di/Dif)": int(((df["Df"] < 0) | (df["Di"] < 0) | (df["Dif"] < 0)).sum()),
    "Non-positive density": int((df["density [g/cm^3]"] <= 0).sum()),
    "POAVF outside [0, 1]": int(((df["POAVF"] < 0) | (df["POAVF"] > 1)).sum()),
    "PONAVF outside [0, 1]": int(((df["PONAVF"] < 0) | (df["PONAVF"] > 1)).sum()),
    "Negative CO2 uptake": int((df[TARGET_COL] < 0).sum()),
}
print("Physical validity violation counts:")
for k, v in physical_checks.items():
    print(f"  - {k}: {v}")

# IQR outlier summary for structural features & target
iqr_rows = []
for col in structural_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[col] < low) | (df[col] > high)).sum())
    iqr_rows.append({
        "feature": col,
        "min": round(float(df[col].min()), 4),
        "max": round(float(df[col].max()), 4),
        "IQR_lower": round(float(low), 4),
        "IQR_upper": round(float(high), 4),
        "IQR_outliers": n_out,
        "outlier_pct": round(100.0 * n_out / len(df), 2),
    })
iqr_df = pd.DataFrame(iqr_rows)
print("\nIQR Outlier Summary (retained because all values are physically valid MOFs):")
iqr_df

Physical validity violation counts:
  - Negative surface area (ASA): 0
  - Negative pore diameter (Df/Di/Dif): 0
  - Non-positive density: 0
  - POAVF outside [0, 1]: 0
  - PONAVF outside [0, 1]: 0
  - Negative CO2 uptake: 0

IQR Outlier Summary (retained because all values are physically valid MOFs):


,feature,min,max,IQR_lower,IQR_upper,IQR_outliers,outlier_pct
0,ASA [m^2/cm^3],0.0000,3270.1300,-2320.5225,3867.5375,0,0.00
1,Df,0.0000,71.5020,-0.1557,9.3021,283,5.66
2,Di,0.0000,71.6412,0.2066,12.6958,321,6.42
3,Dif,0.0000,71.6412,0.2355,12.4983,323,6.46
4,NASA [m^2/cm^3],0.0000,1594.4500,-276.5475,460.9125,599,11.98
5,POAV [cm^3/g],0.0000,15.0986,-0.7182,1.1970,215,4.30
6,POAVF,0.0000,0.9611,-0.7519,1.2532,0,0.00
7,PONAV [cm^3/g],0.0000,0.9656,-0.1208,0.2014,380,7.60
8,PONAVF,0.0000,0.5818,-0.1949,0.3248,209,4.18
9,density [g/cm^3],0.0637,4.4706,0.1957,2.3375,143,2.86


## 4. Constant-Column Audit & Data Dictionary Generation

In [5]:
# Identify zero-variance columns (important for documentation & Member 2's feature selection)
constant_cols = [c for c in df.columns if df[c].nunique() <= 1]
print(f"Constant (zero-variance) columns in clean data: {len(constant_cols)}")

# Build comprehensive data_dictionary.md
STRUCTURAL_META = {
    "MOFname": ("Crystal / MOF identifier", "string", "N/A", "Unique refcode string", "Identifier only; excluded from ML features."),
    "ASA [m^2/cm^3]": ("Accessible Surface Area (volumetric)", "float64", "m^2/cm^3", ">= 0", "Geometric accessible surface area per unit volume."),
    "Df": ("Largest Free Sphere diameter (Pore Limiting Diameter, PLD)", "float64", "Angstrom (assumed)", ">= 0", "1 missing row in raw data (FIRNAX01_clean) dropped."),
    "Di": ("Largest Included Sphere diameter (Largest Cavity Diameter, LCD)", "float64", "Angstrom (assumed)", ">= 0", "Maximum spherical cavity diameter."),
    "Dif": ("Largest Included Sphere along Free Sphere path", "float64", "Angstrom (assumed)", ">= 0", "Characteristic channel diameter along diffusion path."),
    "NASA [m^2/cm^3]": ("Non-Accessible Surface Area (volumetric)", "float64", "m^2/cm^3", ">= 0", "Internal surface area unreachable by probe sphere."),
    "POAV [cm^3/g]": ("Probe-Occupiable Accessible Pore Volume (gravimetric)", "float64", "cm^3/g", ">= 0", "Gravimetric accessible void volume."),
    "POAVF": ("Probe-Occupiable Accessible Volume Fraction", "float64", "dimensionless", "[0, 1]", "Volumetric fraction of accessible pores."),
    "PONAV [cm^3/g]": ("Probe-Occupiable Non-Accessible Pore Volume (gravimetric)", "float64", "cm^3/g", ">= 0", "Gravimetric non-accessible void volume."),
    "PONAVF": ("Probe-Occupiable Non-Accessible Volume Fraction", "float64", "dimensionless", "[0, 1]", "Volumetric fraction of non-accessible pores."),
    "density [g/cm^3]": ("Crystal framework mass density", "float64", "g/cm^3", "> 0", "Bulk crystal density of the MOF."),
    "total_SA_volumetric": ("Total volumetric surface area", "float64", "m^2/cm^3 (assumed)", ">= 0", "Check collinearity with ASA + NASA in EDA."),
    "total_SA_gravimetric": ("Total gravimetric surface area", "float64", "m^2/g (assumed)", ">= 0", "Volumetric SA divided by density."),
    "total_POV_volumetric": ("Total volumetric probe-occupiable volume fraction", "float64", "dimensionless", "[0, 1]", "Check collinearity with POAVF + PONAVF in EDA."),
    "total_POV_gravimetric": ("Total gravimetric probe-occupiable pore volume", "float64", "cm^3/g (assumed)", ">= 0", "Check collinearity with POAV + PONAV in EDA."),
    TARGET_COL: ("CO2 adsorption uptake at 0.15 bar and 298 K (Target)", "float64", "mmol/g (assumed)", ">= 0", TARGET_UNIT_NOTE),
}

def describe_rac_column(col: str) -> tuple[str, str, str]:
    """Return (meaning, unit, notes) for Revised Autocorrelation (RAC) descriptors."""
    if "mc_CRY" in col:
        domain = "Metal-centered crystal RAC"
    elif "lc-" in col or "D_lc-" in col:
        domain = "Linker-centered RAC"
    elif "func-" in col or "D_func-" in col:
        domain = "Functional-group RAC"
    elif "f-lig-" in col:
        domain = "Full-ligand RAC"
    else:
        domain = "Chemical RAC descriptor"
    op = "difference " if "D_" in col else ("sum " if col.startswith("sum-") else "product ")
    note = "Zero-variance (constant) in train data." if col in constant_cols else "Chemical graph autocorrelation descriptor."
    return f"{domain} ({op.strip()} descriptor: {col})", "dimensionless", note

DOCS_DIR.mkdir(parents=True, exist_ok=True)
lines = [
    "# Data Dictionary – MOF CO₂ Uptake Prediction\n\n",
    f"- **Primary Labeled Dataset:** `data/raw/train.csv` ({len(raw_train):,} raw rows → {len(df):,} cleaned rows, {len(df.columns)} columns)\n",
    f"- **Unlabeled External File:** `data/raw/test.csv` ({len(raw_test):,} rows, {raw_test.shape[1]} columns; excludes target and 54 constant RACs)\n",
    f"- **Target Column:** `{TARGET_COL}` — {TARGET_UNIT_NOTE}\n\n",
    "| Column Name | Meaning | Data Type | Unit | Observed Range | Raw Missing | Notes |\n",
    "| :--- | :--- | :--- | :--- | :--- | ---: | :--- |\n",
]

for col in df.columns:
    dtype_str = str(df[col].dtype)
    miss = int(missing_counts_raw.get(col, 0))
    if col == RAW_NAME_COL:
        meaning, _, unit, obs_range, notes = STRUCTURAL_META[col]
    elif col in STRUCTURAL_META:
        meaning, _, unit, _, notes = STRUCTURAL_META[col]
        obs_range = f"{df[col].min():.4g} to {df[col].max():.4g}"
    else:
        meaning, unit, notes = describe_rac_column(col)
        obs_range = f"{df[col].min():.4g} to {df[col].max():.4g}"
    lines.append(f"| `{col}` | {meaning} | `{dtype_str}` | {unit} | {obs_range} | {miss} | {notes} |\n")

dict_path = DOCS_DIR / "data_dictionary.md"
dict_path.write_text("".join(lines), encoding="utf-8")
print(f"Saved data dictionary ({len(df.columns)} columns) to: {dict_path}")

Constant (zero-variance) columns in clean data: 54
Saved data dictionary (328 columns) to: C:\Users\nourh\.gemini\antigravity\scratch\mof-co2-uptake-prediction\docs\data_dictionary.md


## 5. Add `row_id`, Save `clean_data.csv`, and Create Fixed Splits (`train`, `val`, `test`)

In [6]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Add row_id BEFORE splitting (Shared Rule & Section 5 requirement)
df.insert(0, ID_COL, np.arange(len(df), dtype=int))
df.to_csv(CLEAN_DATA_PATH, index=False)
print(f"Saved clean_data.csv: {df.shape} -> {CLEAN_DATA_PATH}")

# 70% train / 15% val / 15% test split with RANDOM_SEED=42
assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-9
train_df, temp_df = train_test_split(
    df, test_size=(1.0 - TRAIN_RATIO), random_state=RANDOM_SEED, shuffle=True
)
val_fraction_of_temp = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
val_df, test_df = train_test_split(
    temp_df, test_size=(1.0 - val_fraction_of_temp), random_state=RANDOM_SEED, shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(TRAIN_PATH, index=False)
val_df.to_csv(VAL_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)

# Mandatory Integrity Verification
assert list(train_df.columns) == list(val_df.columns) == list(test_df.columns), "Column mismatch across splits!"
assert TARGET_COL in train_df.columns and TARGET_COL in val_df.columns and TARGET_COL in test_df.columns
train_ids, val_ids, test_ids = set(train_df[ID_COL]), set(val_df[ID_COL]), set(test_df[ID_COL])
assert train_ids.isdisjoint(val_ids) and train_ids.isdisjoint(test_ids) and val_ids.isdisjoint(test_ids), "row_id overlap!"
assert len(train_ids) + len(val_ids) + len(test_ids) == len(df)

print(f"train.csv: {train_df.shape} ({len(train_df)/len(df):.1%})")
print(f"val.csv  : {val_df.shape} ({len(val_df)/len(df):.1%})")
print(f"test.csv : {test_df.shape} ({len(test_df)/len(df):.1%})")
print("All split verification checks PASSED!")

C:\Users\nourh\AppData\Local\Temp\ipykernel_2480\3346652235.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(0, ID_COL, np.arange(len(df), dtype=int))


Saved clean_data.csv: (4999, 329) -> C:\Users\nourh\.gemini\antigravity\scratch\mof-co2-uptake-prediction\data\processed\clean_data.csv


train.csv: (3499, 329) (70.0%)
val.csv  : (750, 329) (15.0%)
test.csv : (750, 329) (15.0%)
All split verification checks PASSED!
